# B2-019 — Session 2: Self-Attention and Masks

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260808`  
**Qualified Book 1 prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C11-neural-training`  
**Remediation:** review the linked Book 1 units before continuing: [book1:F1-scientific-python](../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../book1/units/C11-neural-training/lesson.ipynb).


## 1. Self-attention shape contract

Self-attention forms Q, K, and V from the same sequence representation $X\in\mathbb R^{B\times n\times d}$. The score shape is $(B,n,n)$, and row $i$, column $j$ measures how destination $i$ attends to source $j$.

**Checkpoint 1A.** What shape is `K.transpose(-2,-1)`?

**Checkpoint 1B.** Why can the output preserve sequence length?

In [ ]:
import numpy as np
SEED = 20260808
ATOL = 1e-10
RTOL = 1e-10
def causal_allowed(n):
    return np.tril(np.ones((n, n), dtype=bool))
assert causal_allowed(3).tolist() == [[True, False, False], [True, True, False], [True, True, True]]

## 2. Additive masks before softmax

A Boolean valid-entry mask becomes additive bias 0 for allowed entries and a very negative value for forbidden entries. Add that bias to scaled scores before softmax.

**Checkpoint 2A.** Why is multiplying weights by zero after softmax incorrect?

**Checkpoint 2B.** What must happen if an entire query row is invalid?

In [ ]:
scores = np.array([[0., np.log(3.), 10.]])
valid = np.array([[True, True, False]])
masked = np.where(valid, scores, -np.inf)
weights = np.exp(masked - np.max(masked, axis=-1, keepdims=True))
weights /= weights.sum(axis=-1, keepdims=True)
assert np.allclose(weights, [[0.25, 0.75, 0.0]], atol=ATOL, rtol=RTOL)

## 3. Padding masks

A padding mask suppresses key positions that are placeholders. A shape $(B,n)$ key-valid mask broadcasts to $(B,1,n)$, applying the same source validity to each query. Query padding may also be suppressed at the output boundary.

**Checkpoint 3A.** Which axis represents candidate keys?

**Checkpoint 3B.** Does padding imply a triangular pattern?

## 4. Causal masks

A causal mask permits source $j$ for destination $i$ exactly when $j\le i$. The lower triangular matrix therefore blocks future information.

**Worked example 1.** For length 3, allowed columns are `{0}`, `{0,1}`, and `{0,1,2}`.

**Checkpoint 4A.** How many allowed cells are there at length $n$?

**Checkpoint 4B.** What is the first row's distribution if its only allowed score is finite?

## 5. Mask audit and pitfalls

**Worked example 2.** A row with pre-mask scores $(0,\log 3,10)$ and valid entries `(True, True, False)` becomes weights $(1/4,3/4,0)$.

**Common pitfalls.** Broken: mask after softmax, leaving allowed weights summing below one. Fix: mask scores, then normalize. Broken: reverse the triangle. Fix: explicitly test the first and last rows.

**Exam connections.** A mask question may hide the forbidden entry behind an extreme score; audit exact zero weights.

**Going deeper.** Session 4 packages the same rule inside a PyTorch module.

Checkpoint answers: 1A $(B,d_k,n)$; 1B one output per query; 2A normalization is wrong; 2B reject or define explicitly; 3A last axis; 3B no; 4A $n(n+1)/2$; 4B weight 1 on column 0.